In [1]:
# Load data (if not already loaded)
# Uncomment if needed:
import sys
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("DataLoad").getOrCreate()
df_2018 = spark.read.format("parquet").load("0917_2017_18_with_2017_cost.parquet")
df_og = df_2018.toPandas()

import importlib
import model_pipeline
importlib.reload(model_pipeline)
import pandas as pd
import numpy as np

BIN_FLAG_COLUMNS = model_pipeline.get_bin_flag_columns(df_og) +['lab_monitoring_adherent','nephrology_consult_adherent','early_nephrology_referral']
STAGE_COLUMNS = [col for col in df_og.columns if "stage" in col.lower()]
#["stage_2017",'2017Q1_max_ckd_stage','2017Q2_max_ckd_stage', '2017Q3_max_ckd_stage','2017Q4_max_ckd_stage']
CAT_COLUMNS = df_og.select_dtypes(include=["object","category"]).columns.tolist()
TRUE_NUM_COLUMNS = model_pipeline.get_true_num_columns(df_og,CAT_COLUMNS)+[ 'util_2017', 'total_increasing_quarters_2017'
, 'total_lab_tests', 'ckd_visit_count', 'quarters_with_labs', 'nephrology_visit_count', 'days_to_nephrology','MEDIAN_INCOME']
COST_COLUMNS = [col for col in df_og.columns if 
            "cost" in col.lower() or 
             "quarterly" in col.lower()  or "increasing" in col.lower()
             ]
UTILIZATION_COLUMNS = [col for col in df_og.columns if "claims" not in col.lower() ] + ['util_2017']
print("categorical cols: ", CAT_COLUMNS)
print("stage cols: ", STAGE_COLUMNS)
print(COST_COLUMNS)
leftover_cols = [
    c for c in df_og.columns 
    if c not in CAT_COLUMNS and c not in TRUE_NUM_COLUMNS and c not in STAGE_COLUMNS and c not in BIN_FLAG_COLUMNS 
]

print(f"Number of leftover columns: {len(leftover_cols)}")
print(leftover_cols, df_og.shape)  # preview first 50

def make_cost_stratum_3class(df):
    # Default to low-cost (class 0)
    cost_stratum = pd.Series(0, index=df.index)
    cost_stratum[(df['highcost_gt_50000'] == 1) & (df['highcost_gt_100000'] == 0)] = 1
    # Emergent high cost (class 2): 100k to 200k
    cost_stratum[(df['highcost_gt_100000'] == 1) & (df['highcost_gt_200000'] == 0)] = 2
    # High cost (class 2): 200k+
    cost_stratum[df['highcost_gt_200000'] == 1] = 3
    return cost_stratum

# Add the new column to your data
df_og['cost_stratum_2018'] = make_cost_stratum_3class(df_og)
print(df_og["cost_stratum_2018"].value_counts(dropna=False))

cutoff_columns = [col for col in df_og.columns if col.startswith('highcost_gt_')]

feature_cols = [c for c in df_og.columns
              if c not in  (['annual_cost_2017','annual_cost_2018_deflated',"ENROLID", "cost_stratum_2018"] 
              + cutoff_columns)]    # keep only predictors excl. cost of 2018 and cutoff of 2017
numeric_cols = df_og[feature_cols + ["cost_stratum_2018"]].select_dtypes(include=["number"]).columns
corrs = df_og[numeric_cols].corr()["cost_stratum_2018"].abs().sort_values(ascending=False)
# Columns to drop
high_corr_cols = corrs[corrs > 0.5].index.tolist()
# Remove the target column itself, if present
high_corr_cols = [col for col in high_corr_cols if col != "cost_stratum_2018"]
# Final filtered feature set
feature_cols = [col for col in feature_cols if col not in high_corr_cols]
print("High corr features dropped from prediction columns: ",high_corr_cols)
target_col = "highcost_gt_200000"

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/17 10:01:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/17 10:01:14 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


categorical cols:  ['ENROLID', 'INCOME_LEVEL', 'AGEGRP', 'SEX', 'REGION', 'cost_pattern_2017', 'cost_stability_2017', 'lab_monitoring_intensity']
stage cols:  ['stage_2017', '2017Q1_max_ckd_stage', '2017Q2_max_ckd_stage', '2017Q3_max_ckd_stage', '2017Q4_max_ckd_stage']
['annual_cost_2017', 'highcost_gt_50000_2017', 'highcost_gt_75000_2017', 'highcost_gt_100000_2017', 'highcost_gt_200000_2017', 'highcost_gt_300000_2017', 'highcost_gt_400000_2017', 'highcost_gt_500000_2017', 'annual_cost_2018_deflated', 'highcost_gt_50000', 'highcost_gt_75000', 'highcost_gt_100000', 'highcost_gt_200000', 'highcost_gt_300000', 'highcost_gt_400000', 'highcost_gt_500000', '2017Q1_ckd_cost', '2017Q1_direct_ckd_cost', '2017Q1_procedure_ckd_cost', '2017Q1_comorbidity_ckd_cost', '2017Q2_ckd_cost', '2017Q2_direct_ckd_cost', '2017Q2_procedure_ckd_cost', '2017Q2_comorbidity_ckd_cost', '2017Q3_ckd_cost', '2017Q3_direct_ckd_cost', '2017Q3_procedure_ckd_cost', '2017Q3_comorbidity_ckd_cost', '2017Q4_ckd_cost', '2017Q4

In [2]:
# Split data into train/test/val (same as multiobjective_bilevel.ipynb)
train_ids, test_ids, train_pd, test_pd = model_pipeline.train_test_split_enrol(
    df_og,
    target_col=target_col,
    test_size=0.3,
    verbose=False,
    random_state=123
)
print(f"Train shape: {train_pd.shape}, Test shape: {test_pd.shape}")
print("Feature cols:", len(feature_cols))

val_ids, test_ids, val_pd, test_pd = model_pipeline.train_test_split_enrol(
    test_pd, 
    target_col=target_col,
    test_size=0.5,
    verbose=False
)
X_test = test_pd[feature_cols]
y_test = test_pd[target_col]
X_val = val_pd[feature_cols]
y_val = val_pd[target_col]

print(f"Train: {train_pd.shape}, Val: {val_pd.shape}, Test: {test_pd.shape}")
print(f"Train target distribution:\n{train_pd[target_col].value_counts()}")


Train shape: (23479, 96), Test shape: (10063, 96)
Feature cols: 78
Train: (23479, 96), Val: (5031, 96), Test: (5032, 96)
Train target distribution:
highcost_gt_200000
0    22819
1      660
Name: count, dtype: int64


In [3]:
# Import Column Generation sampler
import sys
sys.path.append('balancing_functions')
import balancing_functions.pushpull_columngeneration_sampler as ppcg_module
importlib.reload(ppcg_module)
from balancing_functions.pushpull_columngeneration_sampler import PushPullSamplerCG

# Prepare cases and controls
maj = train_pd[train_pd[target_col] == 0].copy()
minr = train_pd[train_pd[target_col] == 1].copy()

print(f"Cases (minority): {len(minr)}")
print(f"Controls (majority): {len(maj)}")


Cases (minority): 660
Controls (majority): 22819


In [4]:
# PROTOTYPE: Test Column Generation sampler with ratio=1
# This is a careful test to see if CG is working

print("="*60)
print("PROTOTYPE: Testing Column Generation Push-Pull Sampler")
print("="*60)

# NOTE: The CG sampler's _extract_X method doesn't handle categorical columns
# the same way as the MILP sampler. For now, we'll use a workaround by
# preprocessing the dataframes to only include numeric columns.
# TODO: Update CG sampler to use same preprocessing as MILP version

# Preprocess: Use the same preprocessing as MILP sampler for consistency
# We'll create a temporary sampler instance just for preprocessing
from balancing_functions.pushpull_sampler import PushPullSampler as MILPSampler
temp_milp = MILPSampler(
    random_state=42,
    binary_group=target_col,
    uid_col="ENROLID",
)

# Get preprocessed features (this handles one-hot encoding, normalization, etc.)
exclude_cols = ["cost_stratum_2018"] + COST_COLUMNS
X_cases_preprocessed, X_controls_preprocessed = temp_milp.get_preprocessed_control_case_features(
    cases=minr,
    controls=maj,
    exclude_cols_matching=exclude_cols,
    verbose=True
)

# Create temporary dataframes with preprocessed numeric features only
# (CG sampler expects DataFrames but will extract numeric arrays)
# We'll create minimal dataframes with just the preprocessed features
minr_preprocessed = pd.DataFrame(X_cases_preprocessed, index=minr.index)
maj_preprocessed = pd.DataFrame(X_controls_preprocessed, index=maj.index)

# Copy over the target column and ENROLID for later merging
minr_preprocessed[target_col] = minr[target_col].values
maj_preprocessed[target_col] = maj[target_col].values
if "ENROLID" in minr.columns:
    minr_preprocessed["ENROLID"] = minr["ENROLID"].values
    maj_preprocessed["ENROLID"] = maj["ENROLID"].values


PROTOTYPE: Testing Column Generation Push-Pull Sampler
>>> Distance computation will use 42 features
5 Categorical features (one-hot encoded): ['INCOME_LEVEL', 'AGEGRP', 'SEX', 'REGION', 'lab_monitoring_intensity']
12 Numeric features (normalized): ['stage_2017', 'util_2017', '2017Q1_ckd_claims', '2017Q1_max_ckd_stage', '2017Q2_ckd_claims', '2017Q2_max_ckd_stage', '2017Q3_ckd_claims', '2017Q3_max_ckd_stage', '2017Q4_ckd_claims', '2017Q4_max_ckd_stage', 'total_lab_tests', 'nephrology_visit_count']
25 Binary features (unchanged): ['has_Hypertension', 'has_Type_2_Diabetes', 'has_Anemia', 'has_Hyperlipidemia', 'has_Acute_Kidney_Failure', 'has_Hyperparathyroidism', 'has_Kidney_Transplant', 'has_Vitamin_D_Deficiency', 'has_Long-term_Drug_Therapy', 'has_Hypothyroidism', 'has_Sleep_Apnea', 'Antihyperlipidemic Drugs, NEC (THRCLS_53)', 'Cardiac, Beta Blockers (THRCLS_51)', 'Cardiac, Calcium Channel (THRCLS_52)', 'Psychother, Antidepressants (THRCLS_69)', 'Cardiac Drugs, NEC (THRCLS_46)', 'Cardia

In [5]:
import pickle
info_prev = pickle.load(open("cg_prototype_results/cg_ratio_1.0_info.pkl", "rb"))
active_prev = info_prev["active_idx"]   # <-- the correct warm-start set
len(active_prev)

660

In [ ]:

# Initialize CG sampler
# Parameters:
#   w: weight on pull term (0.5 = balanced)
#   max_iter: max column generation iterations
#   init_size: initial random set of controls
#   random_state: for reproducibility
#   distance_metric: 'euclidean' (default)

cg_sampler = PushPullSamplerCG(
    w=0.5,
    max_iter=1000,
    init_size=100,  # Start with 10 random controls
    random_state=42,
    distance_metric="euclidean"
)

print(f"\nRunning CG sampler with:")
print(f"  - Cases: {len(minr_preprocessed)}")
print(f"  - Controls: {len(maj_preprocessed)}")
print(f"  - Target ratio: 1.0")
print(f"  - Preprocessed features: {X_cases_preprocessed.shape[1]}")
print(f"  - Weight w: {cg_sampler.w}")

# Run CG sampling on preprocessed dataframes
# Note: exclude_cols_matching should be empty now since we've already preprocessed
undersampled_cg, info_cg = cg_sampler.solve_pushpull_cg(
    df_cases=minr_preprocessed,
    df_controls=maj_preprocessed,
    exclude_cols_matching=[target_col, "ENROLID"]+exclude_cols,  # Only exclude target and ID
    final_ratio=1.0,
    verbose=True,
    candidate_subset_per_iter=None, # Price all candidates (can set to e.g., 1000 for speed)
   # warm_start_active_idx=active_prev, # <-- NEW
)

print("\n" + "="*60)
print("CG Sampling Complete!")
print("="*60)
print(f"Final undersampled dataset shape: {undersampled_cg.shape}")
print(f"Selected {len(info_cg['chosen_controls_idx'])} controls")
print(f"Final LP objective: {info_cg['final_lp_obj']:.4f}")
print(f"\nObjective trace: {info_cg['obj_trace']}")
print(f"F1 (pull) trace: {[f'{x:.4f}' for x in info_cg['f1_trace']]}")
print(f"F2 (push) trace: {[f'{x:.4f}' for x in info_cg['f2_trace']]}")



Running CG sampler with:
  - Cases: 660
  - Controls: 22819
  - Target ratio: 1.0
  - Preprocessed features: 52
  - Weight w: 0.5
[PushPullCG] |cases|=660, |controls|=22819, k_target=660
[Init] Warm-start active controls = 660 (k_target=660)
[RMP] status: 0
[RMP] objective: -3.9280108345686355
[Iter 0] RMP obj=-3.9280, f1=4.0663, f2=11.9223, |J|=660
[Pricing] Evaluating 22159 candidates
[Pricing] Adding control j=1687 (approx new obj=-3.9265)
[RMP] status: 0
[RMP] objective: -3.9170820642471678
[Iter 1] RMP obj=-3.9171, f1=4.0629, f2=11.9151, |J|=661
[CG] Active set size reached k_target; stopping CG iterations.


In [ ]:
# Map back to original dataframes
# The CG sampler returns indices into the preprocessed dataframes,
# but we need to map these back to the original minr/maj dataframes
# since the indices should be the same (we preserved them)

# Reconstruct full undersampled dataframe from original dataframes
# All cases + selected controls
undersampled_full = pd.concat([
    minr,  # All cases
    maj.iloc[info_cg['chosen_controls_idx']]  # Selected controls
], ignore_index=True)

# Verify the results
print("Verification:")
print(f"  - Total samples in undersampled: {len(undersampled_full)}")
print(f"  - Cases (should be all): {len(undersampled_full[undersampled_full[target_col] == 1])}")
print(f"  - Selected controls: {len(undersampled_full[undersampled_full[target_col] == 0])}")
print(f"  - Target was: {len(minr)} cases and {len(minr)} controls (ratio=1)")

# Check target distribution
print(f"\nTarget distribution in undersampled data:")
print(undersampled_full[target_col].value_counts())

# Check cost_stratum distribution (if available)
if "cost_stratum_2018" in undersampled_full.columns:
    print(f"\nCost stratum distribution in undersampled data:")
    print(undersampled_full["cost_stratum_2018"].value_counts())

# Save for inspection
import os
results_dir = "./cg_prototype_results"
os.makedirs(results_dir, exist_ok=True)
save_path = f"{results_dir}/cg_ratio_1.0_maxiter_1000.csv"
undersampled_full.to_csv(save_path, index=False)
print(f"\nSaved to: {save_path}")

# Also save the info dict for analysis
import pickle
info_path = f"{results_dir}/cg_ratio_1.0_maxiter_1000_info.pkl"
with open(info_path, 'wb') as f:
    pickle.dump(info_cg, f)
print(f"Saved info dict to: {info_path}")


Verification:
  - Total samples in undersampled: 1320
  - Cases (should be all): 660
  - Selected controls: 660
  - Target was: 660 cases and 660 controls (ratio=1)

Target distribution in undersampled data:
highcost_gt_200000
1    660
0    660
Name: count, dtype: int64

Cost stratum distribution in undersampled data:
cost_stratum_2018
3    660
0    435
2    136
1     89
Name: count, dtype: int64

Saved to: ./cg_prototype_results/cg_ratio_1.0.csv
Saved info dict to: ./cg_prototype_results/cg_ratio_1.0_info.pkl


In [ ]:
# Quick comparison: Check distances to verify CG is working
from sklearn.metrics import pairwise_distances

# Compute distances from selected controls to cases
X_cases_array = X_cases_preprocessed
X_selected = X_controls_preprocessed[info_cg['chosen_controls_idx']]

# Distance of selected controls to nearest case
D_sel = pairwise_distances(X_selected, X_cases_array)
nearest_to_case = D_sel.min(axis=1)

print("Distance Analysis:")
print(f"  Mean distance (selected controls → nearest case): {nearest_to_case.mean():.4f}")
print(f"  Median distance: {np.median(nearest_to_case):.4f}")
print(f"  Min distance: {nearest_to_case.min():.4f}")
print(f"  Max distance: {nearest_to_case.max():.4f}")

# Compare to random selection
n_selected = len(info_cg['chosen_controls_idx'])
random_indices = np.random.choice(len(maj), size=n_selected, replace=False)
X_random = X_controls_preprocessed[random_indices]
D_random = pairwise_distances(X_random, X_cases_array)
nearest_random = D_random.min(axis=1)

print(f"\nRandom selection comparison:")
print(f"  Mean distance (random controls → nearest case): {nearest_random.mean():.4f}")
print(f"  Median distance: {np.median(nearest_random):.4f}")

print(f"\nCG improvement: {nearest_to_case.mean()-nearest_random.mean():.4f} (negative is better)")

# Check dispersion: pairwise distances among selected controls
if n_selected > 1:
    D_selected_pairs = pairwise_distances(X_selected)
    # Upper triangle (excluding diagonal)
    upper_tri = np.triu(D_selected_pairs, k=1)
    selected_dispersion = upper_tri[upper_tri > 0].mean()
    
    D_random_pairs = pairwise_distances(X_random)
    upper_tri_random = np.triu(D_random_pairs, k=1)
    random_dispersion = upper_tri_random[upper_tri_random > 0].mean()
    
    print(f"\nDispersion (average pairwise distance among selected controls):")
    print(f"  CG selected: {selected_dispersion:.4f}")
    print(f"  Random: {random_dispersion:.4f}")
    print(f"  CG improvement: {selected_dispersion - random_dispersion:.4f} (positive is better)")


Distance Analysis:
  Mean distance (selected controls → nearest case): 4.8218
  Median distance: 4.5366
  Min distance: 1.3169
  Max distance: 17.8976

Random selection comparison:
  Mean distance (random controls → nearest case): 3.3426
  Median distance: 3.2627

CG improvement: 1.4792 (negative is better)

Dispersion (average pairwise distance among selected controls):
  CG selected: 11.9223
  Random: 5.5073
  CG improvement: 6.4150 (positive is better)


In [ ]:
import os, pickle
import model_IAI
importlib.reload(model_IAI)
from model_IAI import evaluate_binary_oct, finetune_oct
ratio_values = [1.0]

In [ ]:
metrics_master_path = f"{results_dir}/metrics_master.csv"

minbuckets_M2 = [50, 100, 120, 150]
cps_M2 = [0.00001, 5e-5, 0.0001, 5e-3]
depths_M2 = [5, 7]
all_metrics = []
for final_ratio in ratio_values:
    print("\n====================================")
    print(f"  Running PUSH–PULL sampling ratio={final_ratio:.2f}")
    print("====================================\n")
    save_path = f"{results_dir}/cg_ratio_{final_ratio:.1f}.csv"

    undersampled_training_data = pd.read_csv(save_path)
    # Train OCT
    balanced_model, balanced_params, _, preprocessor, feature_names = finetune_oct(
        X_train=undersampled_training_data[feature_cols],
        y_train=undersampled_training_data[target_col],
        X_val=X_val,
        y_val=y_val,
        categorical_cols=CAT_COLUMNS,
        numeric_cols=TRUE_NUM_COLUMNS,
        depths=depths_M2,
        minbuckets=minbuckets_M2,
        cps=cps_M2,
    )

    # Evaluate
    metrics = evaluate_binary_oct(
        balanced_model, X_test, y_test, preprocessor, feature_names,
        results_dir = results_dir, ratio = final_ratio
    )

    row = metrics.copy()
    row["ratio"] = final_ratio

    # Append
    pd.DataFrame([row]).to_csv(
        metrics_master_path,
        mode="a",
        header=not os.path.exists(metrics_master_path),
        index=False,
    )

    all_metrics.append(row)



  Running PUSH–PULL sampling ratio=1.00

→ Building preprocessor:
   • OneHotEncoder on: ['INCOME_LEVEL', 'AGEGRP', 'SEX', 'REGION', 'cost_pattern_2017', 'cost_stability_2017', 'lab_monitoring_intensity']
   • StandardScaler on: ['2017Q1_ckd_cost', '2017Q1_ckd_claims', '2017Q1_direct_ckd_cost', '2017Q1_procedure_ckd_cost', '2017Q1_comorbidity_ckd_cost', '2017Q2_ckd_cost', '2017Q2_ckd_claims', '2017Q2_direct_ckd_cost', '2017Q2_procedure_ckd_cost', '2017Q2_comorbidity_ckd_cost', '2017Q3_ckd_cost', '2017Q3_ckd_claims', '2017Q3_direct_ckd_cost', '2017Q3_procedure_ckd_cost', '2017Q3_comorbidity_ckd_cost', '2017Q4_ckd_cost', '2017Q4_ckd_claims', '2017Q4_direct_ckd_cost', '2017Q4_procedure_ckd_cost', '2017Q4_comorbidity_ckd_cost', 'ckd_cost_trend_2017', 'ckd_cost_volatility_2017', 'ckd_cost_deriv_Q1_Q2_2017', 'ckd_cost_deriv_Q2_Q3_2017', 'ckd_cost_deriv_Q3_Q4_2017', 'avg_quarterly_derivative_2017', 'quarterly_std_2017', 'quarterly_skewness_2017', 'quarterly_kurtosis_2017', 'quarterly_cv_2017

/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
┌ Warning: Interpretable AI license expires soon: 2025-12-31T00:00:00. If you need to renew, please send us the following machine ID:
└ da2d87f307115a4389e740fa6b3796f3ca586f217a49634cef5fb2b2945f7cea


Test dataset for OCT application: 5,032 samples


/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


✓ Predictions completed


[ Warning: The type of y (Any) does not match the original target type (Int64)


✓ Saved OCT predictions to: ./cg_prototype_results/predictions/oct_predictions_ratio_1.00.csv
✓ Saved split table (4 splits) to: ./cg_prototype_results/oct_tree_ratio_1.00_splits.csv
AUC score: 0.563
PR-AUC (Average Precision): 0.035
Sensitivity (Recall): 0.655
Specificity: 0.648
Balanced (G-mean) recall: 0.655
Balanced (G-mean) specificity: 0.648
Sensitivity (default): 0.655
Specificity (default): 0.648
Number of leaves: 5
